In [ ]:
# Mount your google drive where you've saved your assignment folder
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!ls

drive  sample_data


In [ ]:
print(f"File size of /content/drive/MyDrive/SBU/Advanced_Project_Robotics/uHumans2_office_s1_00h.bag:")
!ls -lh /content/drive/MyDrive/SBU/Advanced_Project_Robotics/uHumans2_office_s1_00h.bag

File size of /content/drive/MyDrive/SBU/Advanced_Project_Robotics/uHumans2_office_s1_00h.bag:
-rw------- 1 root root 16G Apr 13 03:18 /content/drive/MyDrive/SBU/Advanced_Project_Robotics/uHumans2_office_s1_00h.bag


In [ ]:
pip install rosbags rosbags-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.3 MB/s eta 0:00:00


In [ ]:
pip install rosbags

In [ ]:
import cv2
import csv
import os
import numpy as np
from rosbags.highlevel import AnyReader
from rosbags.image import message_to_cvimage
import torch
from pathlib import Path
import torch.nn.functional as F
from google.colab.patches import cv2_imshow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
drive_path = '/content/drive/MyDrive/SBU/Advanced_Project_Robotics/'
bag_path = drive_path+'uHumans2_office_s1_00h.bag'
updates_bagpath = drive_path+'updated_uHumans2.bag'
results_dir = drive_path+'dataset'
# bagpath = Path('/content/drive/MyDrive/SBU/Advanced_Project_Robotics/bag_pcd_topics.bag')
bagpath = Path(bag_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<h1>Script1: Convert bag file to folders in gdrive</h1>

Images named based on timestamp.

One folder for each topic, where each folder contains respective images and a metadata.csv

In [ ]:
def extract_to_folders(bag_path, base_output_dir):
    bag_path = Path(bag_path)
    with AnyReader([bag_path]) as reader:
        for connection in reader.connections:
            if connection.msgtype == 'sensor_msgs/msg/Image':
                topic_name = connection.topic.replace('/', '_').lstrip('_')
                print(f"Processing topic: {topic_name}")
                topic_dir = Path(base_output_dir) / topic_name
                topic_dir.mkdir(parents=True, exist_ok=True)

                csv_path = topic_dir / "metadata.csv"
                with open(csv_path, 'w', newline='') as csvfile:
                    writer = csv.writer(csvfile)
                    writer.writerow(['filename', 'sec', 'nanosec', 'frame_id', 'encoding'])

                    for conn, timestamp, rawdata in reader.messages(connections=[connection]):
                        msg = reader.deserialize(rawdata, conn.msgtype)
                        
                        if msg.encoding == '32FC1':
                            depth_img = np.frombuffer(msg.data, dtype=np.float32).reshape(msg.height, msg.width)

                            depth_norm = cv2.normalize(depth_img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                            cv_img = depth_norm
                        else:
                            cv_img = message_to_cvimage(msg, 'bgr8')

                        # Use timestamp for unique filename
                        fname = f"{timestamp}.jpg"
                        cv2.imwrite(str(topic_dir / fname), cv_img)

                        writer.writerow([fname, msg.header.stamp.sec,
                                         msg.header.stamp.nanosec,
                                         msg.header.frame_id, msg.encoding])
    print(f"Extraction complete to {base_output_dir}")

extract_to_folders(bag_path, results_dir)


Processing topic: tesse_depth_cam_mono_image_raw
Extraction complete to /content/drive/MyDrive/SBU/Advanced_Project_Robotics/dataset


<h1>Script2: Convert the folders back to bag file</h1>

In [ ]:
from pathlib import Path
import cv2
import csv
from rosbags.rosbag1 import Writer 
from rosbags.typesys import Stores, get_typestore

def folders_to_bag(workspace_dir, output_bag_path):
    typestore = get_typestore(Stores.ROS1_NOETIC)
    Image = typestore.types['sensor_msgs/msg/Image']
    Header = typestore.types['std_msgs/msg/Header']
    Time = typestore.types['builtin_interfaces/msg/Time']

    with Writer(output_bag_path) as writer:
        workspace = Path(workspace_dir)

        # Iterate through every folder in the workspace (each folder is a topic)
        for topic_folder in workspace.iterdir():
            if not topic_folder.is_dir(): continue

            topic_name = "/" + topic_folder.name.replace('_', '/')
            connection = writer.add_connection(topic_name, Image.__msgtype__, typestore=typestore)

            metadata_path = topic_folder / "metadata.csv"
            with open(metadata_path, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    img = cv2.imread(str(topic_folder / row['filename']))

                    # Reconstruct ROS Image Message
                    msg = Image(
                        header=Header(
                            stamp=Time(sec=int(row['sec']), nanosec=int(row['nanosec'])),
                            frame_id=row['frame_id']
                        ),
                        height=img.shape[0],
                        width=img.shape[1],
                        encoding=row['encoding'],
                        is_bigendian=0,
                        step=img.shape[1] * 3,
                        data=img.tobytes()
                    )

                    # Original recording timestamp (nanoseconds)
                    ts = int(Path(row['filename']).stem)
                    writer.write(connection, ts, typestore.serialize_ros1(msg, Image.__msgtype__))

    print(f"Bag created: {output_bag_path}")

folders_to_bag(results_dir, updates_bagpath)
